# 03 — Precipitation Preprocessing

Inventories precipitation rasters and performs basic raster validation.
Actual clipping/reprojection is completed in Notebook 05.

In [1]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

Project root: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [2]:
import pandas as pd
import rasterio

precip_root = RAW_DIR / "precipitation"
raster_files = sorted([
    *precip_root.rglob("*.tif"),
    *precip_root.rglob("*.tiff"),
])

if not raster_files:
    raise FileNotFoundError("No precipitation GeoTIFF files were found.")

records = []
for path in raster_files:
    try:
        with rasterio.open(path) as src:
            records.append({
                "dataset": path.parent.name,
                "file": path.name,
                "crs": str(src.crs),
                "width": src.width,
                "height": src.height,
                "resolution_x": src.res[0],
                "resolution_y": src.res[1],
                "nodata": src.nodata,
                "bands": src.count,
                "status": "OK",
            })
    except Exception as exc:
        records.append({
            "dataset": path.parent.name,
            "file": path.name,
            "status": f"ERROR: {exc}",
        })

precip_inventory = pd.DataFrame(records)
display(precip_inventory.head(20))

,dataset,file,crs,width,height,resolution_x,resolution_y,nodata,bands,status
0,CCS,2017_01.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
1,CCS,2017_02.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
2,CCS,2017_03.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
3,CCS,2017_04.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
4,CCS,2017_05.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
5,CCS,2017_06.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
6,CCS,2017_07.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
7,CCS,2017_08.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
8,CCS,2017_09.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK
9,CCS,2017_10.tif,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,1,OK


In [3]:
output_dir = INTERIM_DIR / "inventories"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "precipitation_raster_inventory.csv"
precip_inventory.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh\data\interim\inventories\precipitation_raster_inventory.csv
